# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (files)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Structured Streaming with Files",
                   master_url="spark://spark-master:7077")

su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 02:35:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Create a data stream from a local socket

### Connect Spark to input folder

In [ ]:
!mkdir -p /opt/spark/work-dir/data/streaming/logs/
!ls /opt/spark/work-dir/data/streaming/logs/

mkdir: cannot create directory ‘/opt/spark/work-dir/data/streaming/logs/’: File exists


In [3]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

input_path = "/opt/spark/work-dir/data/streaming/logs/"

# Create the stream
logs_df = (su.spark.readStream
            .format("text")
            .option("maxFilesPerTrigger", 1) # Let's process one file at a time
            .schema(logs_schema)
            .load(input_path))

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")               # keep only the clean columns
    .filter(F.col("timestamp").isNotNull())  # skip malformed lines
)

# Let's create a summary
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Write stream in the destination
query_events = (
    parsed_df.writeStream
    .outputMode("append")        # append: show new rows only
    .format("console")
    .option("truncate", False)   # don't cut off long messages
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()


26/04/07 02:45:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/07 02:45:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0890627e-b92f-4927-b981-f72a1c18e98c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/07 02:45:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


   Press Ctrl+C to stop.



-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-----+------------------------------------+-------------+
|timestamp          |level|message                             |server       |
+-------------------+-----+------------------------------------+-------------+
|2026-04-07 02:53:56|WARN |Memory usage 90%                    |server-node-1|
|2026-04-07 02:53:59|ERROR|Authentication failed               |server-node-3|
|2026-04-07 02:54:02|ERROR|Unhandled exception in worker thread|server-node-2|
|2026-04-07 02:54:06|WARN |Retry attempt 3 of 5                |server-node-5|
|2026-04-07 02:54:09|INFO |Configuration reloaded              |server-node-4|
|2026-04-07 02:54:13|ERROR|Authentication failed               |server-node-3|
|2026-04-07 02:54:19|INFO |Cache cleared                       |server-node-5|
|2026-04-07 02:54:24|WARN |Certificate expires in 7 days       |server-node-1|
|2026-04-07 02:54:34|ERROR|404 Not

26/04/07 03:08:04 WARN FileStreamSource: Listed 5 file(s) in 2444 ms
26/04/07 03:08:04 WARN FileStreamSource: Listed 5 file(s) in 2444 ms
26/04/07 03:33:11 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/07 03:51:32 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 1046873 ms exceeds timeout 120000 ms
26/04/07 03:51:32 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.8: Executor heartbeat timed out after 1046873 ms
26/04/07 03:51:36 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/04/07 04:51:45 WARN HeartbeatReceiver: Removing executor 3 with no recent heartbeats: 910407 ms exceeds timeout 120000 ms
26/04/07 04:51:45 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.8: Executor heartbeat timed out after 910407 ms
26/04/07 04:51:49 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 se

KeyboardInterrupt: 

In [ ]:
su.spark.stop()

26/04/08 03:27:34 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-701e72d6-7510-4c11-b243-f31deb21b266. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-701e72d6-7510-4c11-b243-f31deb21b266
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1048)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:67)
	at scala.collection.ArrayOps$.foreach$extens